# Forecast Demand per SKU

This notebook builds one model that forecasts demand for all SKUs at once. Instead of raw dates, we turn the time series into features like recent sales and rolling averages, which is a common trick for demand forecasting.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

sales = pd.read_csv("../data/raw/sku_sales.csv", parse_dates=["date"])
sales = sales.sort_values(["sku_id", "date"])

## Build Features

For each SKU, we look back at sales from 7 and 14 days ago, plus a rolling 7 day average. These lag features give the model something to learn from.

In [2]:
sales["lag_7"] = sales.groupby("sku_id")["units_sold"].shift(7)
sales["lag_14"] = sales.groupby("sku_id")["units_sold"].shift(14)
sales["rolling_mean_7"] = sales.groupby("sku_id")["units_sold"].shift(1).rolling(7).mean()
sales["day_of_week"] = sales["date"].dt.dayofweek

# drop rows that do not have enough history yet
sales = sales.dropna()
sales.head()

,date,sku_id,category,units_sold,lag_7,lag_14,rolling_mean_7,day_of_week
14,2023-01-15,SKU-001,Electronics,25,34.0,24.0,21.857143,6
15,2023-01-16,SKU-001,Electronics,16,17.0,44.0,20.571429,0
16,2023-01-17,SKU-001,Electronics,26,16.0,17.0,20.428571,1
17,2023-01-18,SKU-001,Electronics,20,19.0,16.0,21.857143,2
18,2023-01-19,SKU-001,Electronics,20,19.0,35.0,22.000000,3


## Split by Time

We do not shuffle the rows randomly here, because that would let the model see the future. Instead we hold out the last 28 days for testing.

In [3]:
feature_columns = ["lag_7", "lag_14", "rolling_mean_7", "day_of_week"]
target_column = "units_sold"

cutoff_date = sales["date"].max() - pd.Timedelta(days=28)
train_data = sales[sales["date"] <= cutoff_date]
test_data = sales[sales["date"] > cutoff_date]

print("train rows:", len(train_data))
print("test rows:", len(test_data))

train rows: 7160
test rows: 560


## Train the Model

In [4]:
model = lgb.LGBMRegressor(n_estimators=200, random_state=42, verbosity=-1)
model.fit(train_data[feature_columns], train_data[target_column])

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


## Check Accuracy

WMAPE is a common metric for demand forecasting. It reads as the average error as a percent of total demand, and it handles low volume SKUs better than plain MAPE.

In [5]:
predictions = model.predict(test_data[feature_columns])
actual = test_data[target_column].values

mae = mean_absolute_error(actual, predictions)
wmape = np.sum(np.abs(actual - predictions)) / np.sum(np.abs(actual)) * 100

print("MAE:", round(mae, 2))
print(f"WMAPE: {wmape:.1f}%")

MAE: 6.21
WMAPE: 14.1%


## Save Per SKU Demand Stats

The safety stock notebook needs the average daily demand and its standard deviation for each SKU.

In [6]:
sku_stats = sales.groupby("sku_id")["units_sold"].agg(["mean", "std"]).reset_index()
sku_stats.columns = ["sku_id", "avg_daily_demand", "demand_std"]

sku_stats.to_csv("../data/raw/sku_stats.csv", index=False)
sku_stats.head()

,sku_id,avg_daily_demand,demand_std
0,SKU-001,26.386010,9.446979
1,SKU-002,38.082902,7.734783
2,SKU-003,24.652850,8.494380
3,SKU-004,36.860104,9.359827
4,SKU-005,35.269430,10.115692
